# 02 — Normalisation et résolution du pays

Ordre de preuve : source ISO normalisée → `(code postal, ville)` non ambigu → `ville` non ambiguë → cache/géocodage → overrides ciblés par `stage_id`. Chaque inférence possède une méthode, un niveau de confiance et, lorsqu'elle provient du dataset, un nombre d'observations de support.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.config import load_csv_config
from stages_cleaning.country import (
    apply_country_geocode_cache, apply_country_overrides, apply_location_overrides,
    correct_source_conflicts_from_geocode, finalize_country, infer_country_from_source,
    prepare_country_columns,
)
from stages_cleaning.geocode import complete_country_cache, location_key

INPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_name.csv"
OUTPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_country.csv"
CACHE_PATH = ROOT / "data" / "interim" / "country_geocoding_cache.csv"
from stages_cleaning.paths import dataset_paths, network_enabled

OVERRIDE_PATH = dataset_paths(ROOT).location_overrides
# Décision d'équipe : RUN_NETWORK_GEOCODING reste à True par défaut (contrairement à la
# recommandation initiale de l'audit round 3, cf. AUDIT_FIXES.md B-1). Le géocodage réseau
# résout une part significative des pays/CP manquants et le cache rend les runs suivants
# hors ligne. Les risques identifiés (reproductibilité, RGPD, cache empoisonné par une panne
# réseau) sont couverts par les correctifs livrés : geocode.py ne met plus jamais une panne
# réseau en cache comme "not_found" (B-2), et les notebooks ne versionnent plus leurs sorties
# nominatives (R-1). Repasser à False pour un run strictement hors ligne / reproductible.
RUN_NETWORK_GEOCODING = network_enabled()  # STAGES_RUN_NETWORK=0 pour un run hors ligne

df = pd.read_csv(INPUT_PATH, dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
overrides = load_csv_config(OVERRIDE_PATH, {"stage_id", "corrected_city", "corrected_country", "reason"})
print("Shape entrée :", df.shape)

## 1. Corrections de localisation tracées et préparation

In [ ]:
df = apply_location_overrides(df, overrides)
df = prepare_country_columns(df)
print(df["country_source_status"].value_counts(dropna=False))
print(df["city_resolution_method"].value_counts(dropna=False))

## 2. Inférences internes avec support et confiance

In [ ]:
df = infer_country_from_source(df)
print(df["country_resolution_method"].value_counts(dropna=False))
print(pd.crosstab(df["country_resolution_method"], df["country_resolution_confidence"], dropna=False))
low_evidence = df[df["country_resolution_confidence"].isin(["low", "medium"])]
display(low_evidence[["stage_id", "ville", "code_postal", "pays_final", "country_resolution_method", "country_resolution_confidence", "country_reference_count"]].head(30))

## 3. Cache et géocodage des résidus

In [ ]:
remaining = df["pays_final"].isna()
locations = df.loc[remaining, ["adresse", "code_postal", "_ville_geocode"]].drop_duplicates().rename(columns={"_ville_geocode": "ville"})
locations["location_key"] = location_key(locations, ["adresse", "code_postal", "ville"])
cache = complete_country_cache(locations, CACHE_PATH, run_network=RUN_NETWORK_GEOCODING)
join_frame = df[["adresse", "code_postal", "_ville_geocode"]].rename(columns={"_ville_geocode": "ville"})
df["location_key"] = location_key(join_frame, ["adresse", "code_postal", "ville"])
df = apply_country_geocode_cache(df, cache)
df = correct_source_conflicts_from_geocode(df)
print("Cache :", len(cache), "lignes | pays encore manquants :", int(df["pays_final"].isna().sum()))

## 4. Overrides ciblés et validation finale

In [ ]:
df = apply_country_overrides(df, overrides)
df = finalize_country(df)
print(df["country_resolution_method"].value_counts(dropna=False))
print(df["country_resolution_confidence"].value_counts(dropna=False))
print("Pays non résolus :", int(df["pays"].isna().sum()))

## 5. Export

In [ ]:
drop_cols = ["pays_source", "pays_final", "_ville_geocode", "_ville_key", "_cp_key", "location_key", "pays_geocode", "geocode_method", "geocode_query"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Export :", OUTPUT_PATH)
print("Shape :", df.shape)